# Image input matrix: every capable model reads the picture, every blind one is refused

Sends known-answer fixtures (a 4-digit token as png, jpeg, gif and webp; a code word in a
text PDF) to each model as url4 attachments, and runs hand-picked gold Cases of a picture
Benchmark through `client.evaluate(..., cases=[...])`. The expected outcome per cell comes from
the local gateway's `/v1/models`: a model listing the modality must answer with the known
value; one that does not must be refused before any paid call. A model the listing does not
describe is skipped, never passed.

Fixtures are fetched from `cdn.jsdelivr.net/gh` at `FIXTURE_REF` and checked against
`manifest.json` before any model call, so a stale ref is a setup error. Local target only:
it spends real tokens (`BUDGET_USD` caps the run). The `workbench-result` cell reports the
matrix before the verdict cell asserts on it.

In [ ]:
MODELS = [
    "openrouter/google/gemini-3-flash-preview",
    "openrouter/openai/gpt-5-nano",
    "openrouter/anthropic/claude-haiku-5.5",
    "openrouter/meta-llama/llama-4-scout",  # images, no PDF
    "openrouter/mistralai/mistral-small-24b-instruct-2501",  # text-only control
]
MEDIA = ["image/png", "image/jpeg", "image/gif", "image/webp", "application/pdf"]
PATHS = ["url4", "benchmark"]
FIXTURE_REF = "6c127c630615b08fa4435fb1b130392606243bbc"  # the commit that added the fixtures
BUDGET_USD = 0.5
ENGINE_URL = "http://127.0.0.1:9108"
SCOREBOARD_URL = "http://127.0.0.1:9106"  # always explicit, never the hosted leaderboard
GATEWAY_URL = "http://127.0.0.1:9105"
EVENTS_PATH = ""  # the runtime.log holding the gateway's lines; the local target injects it

In [ ]:
import hashlib
import json
import os
import sys
from dataclasses import replace
from decimal import Decimal
from pathlib import Path

import httpx
from IPython.display import HTML, display

import screamingface as sf

E2E_ROOT = Path.cwd().resolve().parents[1]
if str(E2E_ROOT) not in sys.path:
    sys.path.insert(0, str(E2E_ROOT))

from workbench.image_matrix import expectation, verdict  # noqa: E402
from workbench.image_notebook import (  # noqa: E402
    BENCHMARK_CASE_ESTIMATE_USD,
    BENCHMARK_MEDIA,
    PROMPT,
    URL4_CALL_ESTIMATE_USD,
    Budget,
    benchmark_run_cells,
    coverage_failures,
    html_table,
    matrix_rows,
    reported_trace_ids,
    skipped_cell,
    upstream_calls,
    url4_cell,
)
from workbench.image_url4 import fixture_url, run_attachment, verify_fixture  # noqa: E402

FIXTURES = E2E_ROOT / "workbench" / "fixtures" / "image_input"

In [ ]:
if not SCOREBOARD_URL:
    raise ValueError("SCOREBOARD_URL must be explicit")
if not EVENTS_PATH or not Path(EVENTS_PATH).is_file():
    raise ValueError(f"EVENTS_PATH must name the local runtime.log, got {EVENTS_PATH!r}")
client = sf.Client(engine_url=ENGINE_URL, scoreboard_url=SCOREBOARD_URL)
client.connect("openrouter", api_key=os.environ["OPENROUTER_API_KEY"])
http = httpx.Client(timeout=600)
listing = http.get(f"{GATEWAY_URL}/v1/models").raise_for_status().json()
manifest = json.loads((FIXTURES / "manifest.json").read_text(encoding="utf-8"))
fixtures = [entry for entry in manifest if entry["media_type"] in MEDIA]
for entry in fixtures:
    entry["url"] = fixture_url(FIXTURE_REF, entry["path"])
    verify_fixture(http, entry["url"], entry["sha256"])  # FixtureError names the URL
gold = json.loads((FIXTURES / "gold_cases.json").read_text(encoding="utf-8"))
budget = Budget(Decimal(str(BUDGET_USD)))
cells = []
checked_cases = []
media_failures = []
print(f"{len(fixtures)} fixtures verified at {FIXTURE_REF}; {len(gold)} gold Cases")

In [ ]:
for model in MODELS if "url4" in PATHS else []:
    for entry in fixtures:
        media_type = entry["media_type"]
        expected = expectation(listing, model, media_type)
        if expected == "unknown":
            cells.append(
                skipped_cell("url4", model, media_type, expected, "not in the gateway listing")
            )
            continue
        if not budget.allows(URL4_CALL_ESTIMATE_USD):
            why = f"budget: ${budget.spent} of ${budget.limit} spent"
            cells.append(skipped_cell("url4", model, media_type, expected, why))
            continue
        outcome = run_attachment(http, ENGINE_URL, model, entry["url"], media_type, PROMPT)
        cell = url4_cell(model, media_type, expected, outcome, entry["expected"], chat_calls=None)
        if cell.observed != "refused":
            budget.charge(outcome.spend_usd)
        cells.append(cell)
        print(f"url4 {model} {media_type}: {cell.observed} ({cell.code or 'ok'})")

In [ ]:
by_benchmark = {}
for entry in gold:
    by_benchmark.setdefault(entry["benchmark"], []).append(entry)
for model in MODELS if "benchmark" in PATHS else []:
    expected = expectation(listing, model, BENCHMARK_MEDIA)
    for benchmark, golds in by_benchmark.items():
        run_golds = golds[:1] if expected == "refuse" else golds
        projected = BENCHMARK_CASE_ESTIMATE_USD * len(run_golds)
        why = None
        if expected == "unknown":
            why = "not in the gateway listing"
        elif not budget.allows(projected):
            why = f"budget: ${budget.spent} of ${budget.limit} spent"
        if why is not None:
            for entry in run_golds:
                label = f"{benchmark}#{entry['case']}"
                cells.append(
                    skipped_cell("benchmark", model, BENCHMARK_MEDIA, expected, why, case=label)
                )
            continue
        report = None
        error = None
        try:
            report = client.evaluate(
                sf.Model(model),
                benchmark=benchmark,
                cases=[entry["case"] for entry in run_golds],
                progress=False,
            )
        except sf.ScreamingFaceError as exc:
            error = exc
            report = getattr(exc, "partial_report", None)
        run_cells, run_cases = benchmark_run_cells(
            model, benchmark, run_golds, expected, report, error
        )
        budget.charge(run_cells[0].spend_usd, estimate=projected)
        cells += run_cells
        checked_cases += [(model, case) for case in run_cases]
        for cell in run_cells:
            print(f"benchmark {model} {cell.case}: {cell.observed} ({cell.code or 'ok'})")

In [ ]:
for model, case in checked_cases:
    where = f"benchmark {model} case {case.case_id}"
    if not case.media:
        media_failures.append(f"{where}: the Case Result names no picture")
    for media in case.media:
        try:
            data = client.fetch_media(media).data
        except sf.ScreamingFaceError as exc:
            media_failures.append(f"{where}: fetch_media failed ({exc.code})")
            continue
        if hashlib.sha256(data).hexdigest() != media.sha256:
            media_failures.append(f"{where}: fetched media digest differs from {media.sha256}")

In [ ]:
gateway_log = Path(EVENTS_PATH).read_text(errors="replace")
cells = [
    replace(cell, chat_calls=upstream_calls(gateway_log, cell.trace_id)) if cell.trace_id else cell
    for cell in cells
]
display(HTML(html_table(cells)))
print(f"spend: ${budget.spent} of ${budget.limit} (url4 calls at their estimate)")

In [ ]:
print(
    "WORKBENCH_RESULT="
    + json.dumps({"trace_ids": reported_trace_ids(cells), "matrix": matrix_rows(cells)})
)

In [ ]:
failures = verdict(cells) + coverage_failures(cells) + media_failures
assert not failures, "\n".join(failures)

In [ ]:
http.close()
client.close()